# Human iEEG coding dimension, timescale and coupling: Kauvar, Richman, Liu et al. (2025), dandiset 001347

Paper. Kauvar IV, Richman EB, Liu TX, et al. (2025). Conserved brain-wide emergence of emotional
response from sensory experience in humans and mice. *Science* 388, eadt3971.
[doi:10.1126/science.adt3971](https://doi.org/10.1126/science.adt3971). The claims file was written
from the version of record (29 May 2025) with Materials and Methods, figure legends and the
supplement, including Table S3.

Dandiset. [DANDI:001347](https://dandiarchive.org/dandiset/001347/0.250528.0702), version
0.250528.0702, the only published version. The paper cites the dandiset without a version, so the
latest published version is pinned (see `dandi_io.py`). The mouse Neuropixels data used for
figs37a are in [DANDI:001326](https://dandiarchive.org/dandiset/001326/0.250528.1957), version
0.250528.1957, its only published version.

What this notebook reads. For the 7 included iEEG participants (SD168, SD172, SD181, SD185,
SD186, SD211, SD245), the per-session files `sub-SD*/sub-SD*_ses-puff<Session>_ecephys.nwb`
(LFP at 200 Hz, 20 min around the eyepuff assay, an electrode table with region labels, and 20 DC
channels, one of which carries the puff TTL) and the behaviour files
`sub-SD*/sub-SD*_ses-puff_behavior.nwb` (trials used, puff frames of the video, per-trial eye
closure). For figs37a, the 13 Neuropixels sessions of DANDI:001326, processed with the entry's
`mouse_npx.py`.

Outside input. The paper's code record
[Zenodo 10.5281/zenodo.14920570](https://doi.org/10.5281/zenodo.14920570)
(`kauvar-richman-liu-2025_zenodo.zip`, md5 bd1fb567121aaf0291337783df57bcfc, git commit b873ca5)
is fetched at run time and its md5 checked. Two things are read from it: the list of bad channels
per participant (`ketman/analyze/utils.py`, `bad_ch`; the dataset does not mark bad channels), a
method detail shown as one reading against the dataset-only channel set; and the Yeo7 network label
of each bipolar channel (`anat/pts_ch_labels.pkl`), which the dataset lacks and which figs35 needs
(`data: with_labels`, tested by the link claim `ieeg-channel-atlas-link`). The analysis follows the
code (`human_ieeg/scripts/analyze_airpuff_ieeg_cd.py`, `ketman/analyze/trace_utils.py`,
`cd_utils.py`, `for_reference/analyze_airpuff_ieeg_top_cd_network.py`), reimplemented in
`ieeg_cd.py`, whose docstring says where each parameter comes from.

Random numbers: the random-time null of the clock check uses seed 0 per participant, the Yeo7
shuffle uses seed 1.

| Claim | Data | Status |
|---|---|---|
| fig6c-human-persistent-activity-reduced | dataset | tested |
| fig6e-human-fast-activity-unchanged | dataset | tested |
| figs37c-persistent-reduction-across-bands | dataset | tested |
| figs37b-fast-unchanged-across-bands | dataset | tested |
| fig6g-human-timescale-shorter | dataset | tested |
| fig6i-human-coupling-reduced | dataset | tested |
| figs38a-baseline-coupling-not-different | dataset | tested |
| figs38b-ketamine-coupling-loss-larger-at-persistent-sites | dataset | tested |
| figs38h-prepuff-coupling-predicts-persistence | dataset | tested |
| ieeg-channel-atlas-link (link) | with labels | tested |
| figs35-persistent-dimension-distributed | with labels | tested |
| figs37a-beta-most-similar-to-mouse | with inputs | tested |

In [ ]:
import os, time, warnings
os.environ.setdefault("OMP_NUM_THREADS", "1")
import numpy as np
import pandas as pd
import scipy.stats as st
import yaml
import matplotlib.pyplot as plt
from statsmodels.stats.multitest import multipletests

import dandi_io
import ieeg_cd as ic

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)

T0 = time.time()
with open("../../claims/001347/10.1126-science.adt3971.yaml") as f:
    CLAIMS = {c["id"]: c for c in yaml.safe_load(f)["claims"]}

def expected(cid):
    # Expected values of a claim, read from the claims file for comparison only.
    return [(e["quantity"], e["value"], e.get("tolerance")) for e in CLAIMS[cid].get("expected", [])]

ZF = ic.fetch_code_zip()
BAD = ic.code_bad_channels(ZF)
LABELS = ic.code_channel_labels(ZF)
print("code record fetched, md5 checked; bad-channel lists for",
      sum(pt in BAD for pt in ic.PARTICIPANTS), "of 7 participants")

## Data check

Every participant's sessions are listed from the archive, and one file's electrode table is shown
in full. The processing cell then streams the 22 LFP files used (three per participant, plus
SD245's Preinfusion file, which the code does not use for the coding dimension; see below) and
reduces each to what the claims need. It runs the participants in three parallel processes and
takes about 8 to 10 minutes.

In [ ]:
assets = dandi_io.list_assets()
rows = []
for pt in ic.PARTICIPANTS:
    ses = sorted(p.split("_ses-puff")[1].split("_")[0] for p in assets
                 if p.startswith(f"sub-{pt}/") and p.endswith("_ecephys.nwb"))
    beh_file = f"sub-{pt}/sub-{pt}_ses-puff_behavior.nwb" in assets
    rows.append(dict(participant=pt, ecephys_sessions=", ".join(ses), behaviour_file=beh_file,
                     preinfusion_file_code=ic.PRE_SESSION_CODE[pt]))
print(pd.DataFrame(rows).to_string(index=False))

f = dandi_io.open_h5("sub-SD181/sub-SD181_ses-puffPreinfusion_ecephys.nwb")
e = f["general/extracellular_ephys/electrodes"]
print("\nelectrode table columns:", list(e.attrs["colnames"]))
for col in e.attrs["colnames"]:
    if col == "group":
        continue
    v = e[col][:]
    print(f"  {col}: {len(v)} rows, first values {v[:3]}" +
          (f", NaN fraction {np.isnan(v).mean():.2f}" if v.dtype.kind == "f" else ""))
ts = f["processing/ecephys/LFP/v_trace"]
print("LFP:", ts["data"].shape, "unit attribute:", ts["data"].attrs["unit"],
      "| comments:", ts.attrs["comments"], "| rate:", ts["starting_time"].attrs["rate"])
print("DC channels:", [x.decode() for x in f["acquisition/dc_channels/channel_name"][:]])

In [ ]:
t = time.time()
RES, ALT, BEH = ic.run_all(BAD, max_workers=3)
EX = ic.extra_session(RES["SD245"], "Preinfusion")
print(f"processed in {time.time() - t:.0f} s")

In [ ]:
rows = []
for pt in ic.PARTICIPANTS:
    r, b = RES[pt], BEH[pt]
    wt = b["which_trials"]
    for c in ic.CONDITIONS:
        s = r["sessions"][c]
        bkey = ic.BEHAVIOUR_PRE[r["pre_ses"]] if c == "preinfusion" else c
        vid = b["events"][bkey]
        n = min(len(vid), len(s["puffs"]))
        d = np.diff(s["puffs"][:n]) - np.diff(vid[:n])
        rows.append(dict(pt=pt, cond=c, file=s["file_session"], mono=s["n_mono"], bipolar=s["n_pairs"],
                         ttl_puffs=len(s["puffs"]), first_puff_sample=s["puffs"][0],
                         video_puffs=len(vid), max_interval_diff_samples=int(np.abs(d).max()),
                         trials_used=len(wt), max_trial_index=int(wt.max()),
                         raw_p1=round(s["raw_range"][0]), raw_p99=round(s["raw_range"][1]),
                         dc2_name=s["dc_name"]))
chk = pd.DataFrame(rows)
print(chk.to_string(index=False))
print("\ntrials used per participant:", {pt: len(BEH[pt]["which_trials"]) for pt in ic.PARTICIPANTS},
      "total", sum(len(BEH[pt]["which_trials"]) for pt in ic.PARTICIPANTS))

rows = []
for pt in ic.PARTICIPANTS:
    pr, m = RES[pt]["pairs"], RES[pt]["masks"]
    labels = set(pr["a"]) | set(pr["b"])
    rows.append(dict(pt=pt, bipolar_pairs=len(pr), out_of_brain_or_excluded=int(pr["excluded"].sum()),
                     white_matter=int(pr["wm"].sum()),
                     kept_wm_excl=int(m[(True, "none")].sum()), kept_wm_excl_codebad=int(m[(True, "code")].sum()),
                     kept_all=int(m[(False, "none")].sum()), kept_all_codebad=int(m[(False, "code")].sum()),
                     code_bad_listed=len(BAD[pt]), code_bad_found_in_labels=sum(x in labels for x in BAD[pt])))
print(pd.DataFrame(rows).to_string(index=False))

In [ ]:
# Clock check: the puff-locked change (mean over the response window minus the 0.5 s before the
# puff, averaged over trials, absolute value averaged over channels) of the broadband LFP
# (0.05 to 0.5 s) and of beta power (0.05 to 1.5 s), at the TTL puff times, at 20 sets of 40
# random times in the 10 min before the assay, and at the TTL times shifted by -2.5 s and +2.5 s.
rows = []
for pt in ic.PARTICIPANTS:
    for band in ("broadband", "beta"):
        c = RES[pt]["clock"][band]
        rows.append(dict(pt=pt, signal=band, true=c["true"], random_mean=c["null"].mean(), random_max=c["null"].max(),
                         shift_minus2p5=c["shift"][0], shift_plus2p5=c["shift"][1],
                         true_over_random_max=c["true"] / c["null"].max()))
clock = pd.DataFrame(rows)
print(clock.round(3).to_string(index=False))
for band in ("broadband", "beta"):
    g = clock[clock.signal == band]
    print(f"{band}: true value above every random set in {int((g.true > g.random_max).sum())} of 7 participants, "
          f"above both shifted sets in {int(((g.true > g.shift_minus2p5) & (g.true > g.shift_plus2p5)).sum())} of 7")

Checks of the metadata against the data:

- Session files. SD168 and SD172 have no Preinfusion file; their pre-infusion eyepuff session is
  the Baseline file, recorded the day before (`ketman/DATASETS.py`). The code takes the Baseline
  file as "preinfusion" for these two. For SD245 the code lists both Baseline and Preinfusion and
  renames 'baseline' to 'preinfusion' after loading, which replaces the Preinfusion data by the
  Baseline data in the coding-dimension path (`load_and_organize_raw_traces`,
  `load_patients_erp`), while the timescale path (`load_raw_traces_and_merge_to_anat`) keeps the
  Preinfusion file. SD245's behaviour file holds only 'baseline' eye-closure events. Both SD245
  files are run here as readings.
- Puff times. The TTL is on DC channel index 2, named 'DC03' in three participants' files and
  'CCEP' in the other four, so the channel name does not identify it; the code reads index 2. 40
  onsets are found in every preinfusion session (32 in some infusion and postinfusion sessions),
  with the first at sample 120001 to 120012, the middle of the 20-min epoch, as the epoch
  description ("10 min pre and post puff start") says. Their intervals match the puff frames of
  the 200-Hz video within 3 samples, except the first interval of SD172's infusion session (38
  samples; its first trial is not used). The puff-locked broadband response at the TTL times is
  larger than at every random set of times and at the shifted times in every participant (counts
  above). The puff times are consistent with sharing the LFP clock.
- Trials. The trials used (`which_trials` of the behaviour files) total 214 across participants,
  the number Table S3 gives for fig. S38H.
- Units. The LFP's unit attribute is 'volts', but the values are of order 100 to 1000 and the
  series comment and device description give a 'scale_to_microvolts' factor (0.39 or 0.098), so
  the stored values are digitiser units. All analyses here are z-scored, so the scale does not
  matter. The x, y, z columns of the electrode table are all NaN.
- Channels. The dataset gives a region label per contact, including 'WM', 'OUT OF BRAIN',
  'EXCLUDE' and 'EXCLUDE AREA', but no bad-channel flag and no Yeo7 network label.

## fig6c-human-persistent-activity-reduced

> In humans, eyepuff-evoked activity along the persistent (beta band power) dimension falls during
> ketamine (1.36 to 0.41) and partly recovers after infusion (0.72).

Criterion: H-CD with the persistent window, beta band; reproduced when, under a combination of
readings of Open C and Open D, both corrected p < 0.05 in the stated direction and the three means
are within tolerance. The table shows every combination of Open C (Methods or code z-scoring),
Open D (white matter excluded or kept), the bad-channel list (none, or the code's), the code's
per-trial baseline subtraction (with or without) and SD245's preinfusion file (Baseline as in the
code, or Preinfusion).

In [ ]:
print("expected:", expected("fig6c-human-persistent-activity-reduced"))
T6C = ic.readings_table(RES, ALT, "beta", "persistent")
lo = [v - t for _, v, t in expected("fig6c-human-persistent-activity-reduced")]
hi = [v + t for _, v, t in expected("fig6c-human-persistent-activity-reduced")]
T6C["means_in_tol"] = [(lo[0] <= a <= hi[0]) and (lo[1] <= b <= hi[1]) and (lo[2] <= c <= hi[2])
                       for a, b, c in zip(T6C.pre, T6C.ket, T6C.post)]
T6C["tests_hold"] = ((T6C.pcorr_pre_ket < 0.05) & (T6C.t_pre_ket > 0) &
                     (T6C.pcorr_post_ket < 0.05) & (T6C.t_post_ket > 0))
cols = ["open_c", "wm_excluded", "bad_list", "trial_baseline", "sd245_pre", "pre", "ket", "post",
        "t_pre_ket", "pcorr_pre_ket", "t_post_ket", "pcorr_post_ket", "tests_hold", "means_in_tol"]
print(T6C[cols].round(3).to_string())
print("\nreadings with tests holding:", int(T6C.tests_hold.sum()), "of", len(T6C),
      "| tests and means:", int((T6C.tests_hold & T6C.means_in_tol).sum()))
for key, g in T6C.groupby(["open_c", "bad_list"]):
    print(key, "tests hold", int(g.tests_hold.sum()), "of", len(g), "| tests and means", int((g.tests_hold & g.means_in_tol).sum()))

In [ ]:
# Per participant, the code's readings (Open C code, white matter excluded, code bad list,
# per-trial baseline, SD245 Baseline file), and with the dataset-only channel set.
for bad in ("code", "none"):
    v = ic.cd_summary(RES, ALT, "beta", "persistent", True, bad, True, "code")
    df = pd.DataFrame(v, index=ic.PARTICIPANTS)
    print(f"bad list = {bad}"); print(df.round(3).to_string())
    print("ketamine below preinfusion in", int((df.infusion < df.preinfusion).sum()), "of 7;",
          "postinfusion above ketamine in", int((df.postinfusion > df.infusion).sum()), "of 7")
tr = {c: ic.cd_summary(RES, ALT, "beta", "persistent", True, "code", True, "code", traces=True)[c] for c in ic.CONDITIONS}
fig, ax = plt.subplots(figsize=(4, 2.6))
for c, col in zip(ic.CONDITIONS, ["k", "m", "c"]):
    m, s = tr[c].mean(0), st.sem(tr[c], axis=0)
    ax.plot(ic.TT - 0.05, m, col, label=c); ax.fill_between(ic.TT - 0.05, m - s, m + s, color=col, alpha=0.2)
ax.set_xlim(-1, 2); ax.set_xlabel("time from puff (s, code axis)"); ax.set_ylabel("projection (z)"); ax.legend(fontsize=7)
print("plotted range of the mean traces:", round(min(tr[c].mean(0).min() for c in tr), 2), "to", round(max(tr[c].mean(0).max() for c in tr), 2))
plt.show()

**Result: reproduced under the code's z-scoring (Open C reading b), the paper's numbers recovered.**
With the code's readings (white matter excluded, the code's bad-channel list, SD245 Baseline file)
the means are 1.363, 0.411 and 0.724, with t = 5.876 (corrected p = 0.002) for preinfusion against
ketamine and t = 3.172 (corrected p = 0.019) for postinfusion against ketamine; the paper gives
1.36, 0.41, 0.72, t = 5.887 and 3.171. With white matter kept (Open D b) the three means are also
within tolerance. Without the code's bad-channel list the tests hold in all 8 Open C (b) readings
and the means are within tolerance in 7 of them (with white matter kept, 1.391, 0.472, 0.828; the
exception is white matter excluded with the per-trial baseline, ketamine 0.482). Under the Methods
z-scoring (Open C a) the preinfusion mean is 1.616 to 1.661 in every reading, outside tolerance;
the preinfusion test holds in all 16 of those readings and the postinfusion test in 11. Ketamine
is below preinfusion in all 7 participants.

## fig6e-human-fast-activity-unchanged

> In humans, activity along the fast dimension (0.05 to 0.10 s) is not changed by ketamine (4.12, 4.00, 3.86).

Criterion: a claim of absence judged on the paper's test; reproduced when both corrected p >= 0.05.

In [ ]:
print("expected:", expected("fig6e-human-fast-activity-unchanged"))
T6E = ic.readings_table(RES, ALT, "beta", "fast")
T6E["both_ns"] = (T6E.pcorr_pre_ket >= 0.05) & (T6E.pcorr_post_ket >= 0.05)
cols = ["open_c", "wm_excluded", "bad_list", "trial_baseline", "sd245_pre", "pre", "ket", "post",
        "t_pre_ket", "pcorr_pre_ket", "t_post_ket", "pcorr_post_ket", "both_ns"]
print(T6E[cols].round(3).to_string())
for key, g in T6E.groupby(["open_c", "bad_list"]):
    print(key, "both non-significant in", int(g.both_ns.sum()), "of", len(g))
v = ic.cd_summary(RES, ALT, "beta", "fast", True, "code", True, "code")
d = v["preinfusion"] - v["infusion"]
print("\ncode readings, per participant:"); print(pd.DataFrame(v, index=ic.PARTICIPANTS).round(3).to_string())
print(f"pre - ket mean {d.mean():.3f}, Cohen dz {d.mean() / d.std(ddof=1):.2f}, 95% CI of the mean difference "
      f"{d.mean() - st.t.ppf(0.975, 6) * st.sem(d):.3f} to {d.mean() + st.t.ppf(0.975, 6) * st.sem(d):.3f}")

**Result: reproduced, resting on the code's z-scoring (Open C reading b) where the bad-channel list
is the code's.** Under the code's readings the means are 4.120, 4.006 and 3.866 with t = 0.433
(corrected p = 0.680) for preinfusion against ketamine, matching the paper's 4.12, 4.00, 3.86 and
t = 0.432. Without the code's bad-channel list no reading gives a corrected p below 0.05. With the
code's list and the Methods z-scoring (Open C a), postinfusion against ketamine gives corrected
p = 0.029 to 0.039 (postinfusion higher than ketamine), so the outcome depends on Open C in those
readings; the code's reading is adopted because it reproduces the paper's values. The preinfusion
against ketamine test is not significant in any of the 32 readings (smallest corrected p 0.131).

## figs37c-persistent-reduction-across-bands

> The ketamine reduction of the persistent projection holds in every band except broadband LFP, and
> recovers significantly after infusion in delta and beta.

Criterion: H-CD with the persistent window in each band, under the readings that reproduce fig6c;
reproduced when pre against ketamine is significant in each filtered band (not in broadband) and
post against ketamine is significant in delta and beta; partially reproduced when four of the five
filtered bands hold. Shown with the per-trial baseline subtraction and SD245 Baseline file (the
code's), for each Open C, Open D and bad-channel reading.

In [ ]:
print("expected:", expected("figs37c-persistent-reduction-across-bands"))
rows = []
for band in ic.BANDS:
    t = ic.readings_table(RES, ALT, band, "persistent", sd245s=("code",), bsubs=(True,))
    t.insert(0, "band", band)
    rows.append(t)
T37C = pd.concat(rows, ignore_index=True)
cols = ["band", "open_c", "wm_excluded", "bad_list", "pre", "ket", "post", "t_pre_ket", "pcorr_pre_ket",
        "t_post_ket", "pcorr_post_ket"]
print(T37C[cols].round(3).to_string())
filtered = ["delta", "theta", "alpha", "low gamma", "high gamma"]
print()
for key, g in T37C.groupby(["open_c", "wm_excluded", "bad_list"]):
    g = g.set_index("band")
    sig_pre = [b for b in filtered if g.loc[b, "pcorr_pre_ket"] < 0.05 and g.loc[b, "t_pre_ket"] > 0]
    print(key, "| pre>ket significant in", len(sig_pre), "of 5 filtered bands", sig_pre,
          "| broadband p", round(g.loc["broadband", "pcorr_pre_ket"], 3),
          "| post>ket delta p", round(g.loc["delta", "pcorr_post_ket"], 3), "beta p", round(g.loc["beta", "pcorr_post_ket"], 3))

**Result: reproduced, resting on the white-matter reading (Open D a).** Under the code's readings
(Open C b, white matter excluded) the reduction from preinfusion to ketamine is significant in all
five filtered bands and not in broadband (corrected p = 0.078 with the code's bad-channel list,
0.086 without), and the recovery after infusion is significant in delta (p = 0.047 and 0.048) and
beta. The means (delta 0.793 and 0.177, theta 0.818 and 0.109, alpha 0.972 and 0.249, low gamma
1.316 and 0.490, high gamma 0.898 and 0.267, broadband 1.658 and 1.367 with the code's list) match
Table S3. With white matter kept (Open D b, which also reproduces fig6c), theta (p = 0.10) and the
delta recovery fail, a partial reproduction; under the Methods z-scoring broadband becomes
significant.

## figs37b-fast-unchanged-across-bands

> Activity along the fast dimension does not differ across preinfusion, ketamine and postinfusion in any frequency band.

Criterion: a claim of absence judged on the paper's tests; reproduced when no test in any band gives
corrected p < 0.05.

In [ ]:
rows = []
for band in ic.BANDS:
    t = ic.readings_table(RES, ALT, band, "fast", sd245s=("code",), bsubs=(True,))
    t.insert(0, "band", band)
    rows.append(t)
T37B = pd.concat(rows, ignore_index=True)
print(T37B[cols].round(3).to_string())
print()
for key, g in T37B.groupby(["open_c", "wm_excluded", "bad_list"]):
    sig = g[(g.pcorr_pre_ket < 0.05) | (g.pcorr_post_ket < 0.05)]
    print(key, "| tests with corrected p < 0.05:", len(sig), list(zip(sig.band, sig.pcorr_pre_ket.round(3), sig.pcorr_post_ket.round(3))))

**Result: reproduced, resting on the code's z-scoring (Open C reading b).** Under Open C (b) no band
gives a corrected p below 0.05 in any of the four Open D and bad-channel readings, and the code's
readings give delta t = 0.817 (p = 0.859), as Table S3 does. Under the Methods z-scoring (Open C a)
the low-gamma fast projection is lower during ketamine (corrected p = 0.020 to 0.049) in all four
readings, and with the code's bad-channel list the beta postinfusion test also gives p = 0.034 and
0.039. The code's reading is adopted because it reproduces the paper's values (fig6e and the
delta row).

## fig6g-human-timescale-shorter

> In humans, the baseline intrinsic timescale of the persistent population is shorter during ketamine (0.09 s) than before (0.11 s).

Criterion: H-TAU with the fig6c weights; reproduced when, under a reading of Open E, the paired
t-test gives p < 0.05 with ketamine shorter and both means are within tolerance.

In [ ]:
print("expected:", expected("fig6g-human-timescale-shorter"))
rows, per_pt = [], {}
for wm in (True, False):
    for bad in ("code", "none"):
        for sd in ("Preinfusion", "Baseline"):
            for period in ("10min", "8min"):
                for lag in (1, 0):
                    tau = ic.timescales(RES, EX, wm, bad, period, lag, sd)
                    t, p = st.ttest_rel(tau["preinfusion"], tau["infusion"])
                    rows.append(dict(wm_excluded=wm, bad_list=bad, sd245_pre=sd, period=period, start_lag=lag,
                                     pre=tau["preinfusion"].mean(), ket=tau["infusion"].mean(), t=t, p=p,
                                     n_shorter=int((tau["infusion"] < tau["preinfusion"]).sum())))
                    per_pt[(wm, bad, sd, period, lag)] = tau
T6G = pd.DataFrame(rows)
T6G["holds"] = (T6G.p < 0.05) & (T6G.t > 0)
T6G["means_in_tol"] = (T6G.pre.between(0.10, 0.12)) & (T6G.ket.between(0.08, 0.10))
print(T6G.round(4).to_string())
print("\ncode readings (white matter excluded, code bad list, SD245 Preinfusion file, 10 min, lag 1), per participant:")
print(pd.DataFrame(per_pt[(True, "code", "Preinfusion", "10min", 1)], index=ic.PARTICIPANTS).round(3).to_string())

**Result: reproduced under the code's 10-min reading of Open E.** With the code's readings
(10 min, fit from lag one sample, white matter excluded, the code's bad-channel list, SD245's
Preinfusion file) the timescale is 0.115 s before and 0.092 s during ketamine, t = 4.78,
p = 0.003 (paper 0.11, 0.09, t = 4.74, p = 0.0032), shorter in all 7 participants. The Methods' reading (8 min, fit from lag 0) gives 0.117 s and
0.089 s, p = 0.016, also within tolerance. The test holds (p < 0.05, ketamine shorter in all 7) in
28 of the 32 readings; it fails only with white matter kept, the code's list and SD245's Baseline
file. The means are outside tolerance in every reading that uses SD245's Baseline file (preinfusion
0.122 to 0.143 s).

## fig6i-human-coupling-reduced

> In humans, beta phase locking among the most persistent sites at baseline is lower during ketamine than before (0.979 of preinfusion).

Criterion: H-PLV with the persistent sites of the fig6c weights; reproduced when, under a reading of
Open F, the paired t-test across the 7 participants gives p < 0.05 with ketamine lower and the
ketamine ratio is within tolerance. Open F reading (a), the 25-min windows before and during the
infusion, cannot be run: the dataset holds only the 20-min epochs around each eyepuff session. The
reading run is (b), the 10 min before the eyepuff assay in each session (Table S3), with
non-overlapping 5-s windows.

In [ ]:
print("expected:", expected("fig6i-human-coupling-reduced"))
rows = []
for wm in (True, False):
    for bad in ("code", "none"):
        for sd in ("Baseline", "Preinfusion"):
            r = ic.plv_ratio(RES, EX, wm, bad, sd)
            t, p = st.ttest_rel(np.ones(7), r["ratio"])
            tz, pz = st.ttest_1samp(r["z"], 0)
            rows.append(dict(wm_excluded=wm, bad_list=bad, sd245_pre=sd, ratio_mean=r["ratio"].mean(),
                             ratio_sem=st.sem(r["ratio"]), t=t, p=p, n_lower=int((r["ratio"] < 1).sum()),
                             z_change_mean=r["z"].mean(), t_z=tz, p_z=pz, edges=sum(r["n_edges"])))
T6I = pd.DataFrame(rows)
T6I["holds"] = (T6I.p < 0.05) & (T6I.t > 0)
T6I["ratio_in_tol"] = T6I.ratio_mean.between(0.979 - 0.0061, 0.979 + 0.0061)
print(T6I.round(4).to_string())
r = ic.plv_ratio(RES, EX, True, "code", "Baseline")
print("\nper participant (white matter excluded, code bad list):")
print(pd.DataFrame({"pre_plv": r["pre"], "ket_plv": r["ket"], "ratio": r["ratio"], "z_change": r["z"],
                    "edges": r["n_edges"]}, index=ic.PARTICIPANTS).round(4).to_string())

**Result: partially reproduced.** Beta phase locking among the persistent sites is lower during
ketamine in every reading printed above (paired t-test p < 0.05), but the mean ratio, 0.960 with
the code's channel readings, is below the paper's 0.979 ± 0.0061 in every reading. Open F (a)
could not be run, and the windows, stride and exclusion of epileptic or sleep periods of the
paper are not available.

## figs38a-baseline-coupling-not-different and figs38b-ketamine-coupling-loss-larger-at-persistent-sites

> figs38a: Before infusion, phase locking among high-persistence sites does not differ from that among low-persistence sites in any band.
>
> figs38b: The ketamine decrease in phase locking is larger among high-persistence sites than among low-persistence sites, significantly in the alpha, beta and gamma bands.

Criteria: per band, top and bottom 10% of absolute persistent weight (weights in that band), edges
within each set, PLV regressed on high against low edge identity with a mixed-effects model grouped
by participant, Benjamini-Hochberg over bands. figs38a uses preinfusion PLV (absence: reproduced
when no band gives corrected p < 0.05); figs38b uses the ketamine minus preinfusion change
(reproduced when alpha, beta and gamma give corrected p < 0.05 with larger decreases among high
edges and the beta coefficient is within -0.038 ± 0.019). The paper's figure gives the change in z
units ("Ket - Pre phase locking (z)", code ylabel; Methods z-score each channel's PLV to its
preinfusion window), so both the raw change and the change z-scored by the preinfusion window SD are
shown. Gamma is low gamma (30 to 50 Hz) as in Table S3. Bonferroni (the code's 'pval_bonf') is shown
beside BH.

In [ ]:
print("expected figs38b:", expected("figs38b-ketamine-coupling-loss-larger-at-persistent-sites"))
bands6 = [b for b in ic.BANDS if b != "broadband"]
rows = []
for wm in (True, False):
    for bad in ("code", "none"):
        for y in ("sync_base", "sync_change", "sync_change_z"):
            res_b = []
            for band in bands6:
                df = ic.edge_table(RES, band, wm, bad)
                m = ic.mixed(df, f"{y} ~ top_cd", "top_cd[T.True]")
                res_b.append(dict(wm_excluded=wm, bad_list=bad, outcome=y, band=band, **m))
            g = pd.DataFrame(res_b)
            g["p_bh"] = multipletests(g.p, method="fdr_bh")[1]
            g["p_bonf"] = np.minimum(g.p * len(bands6), 1)
            rows.append(g)
T38 = pd.concat(rows, ignore_index=True)
print(T38.round(4).to_string())

In [ ]:
print("figs38a (preinfusion PLV, high vs low): bands with BH-corrected p < 0.05")
for key, g in T38[T38.outcome == "sync_base"].groupby(["wm_excluded", "bad_list"]):
    s = g[g.p_bh < 0.05]
    print(" ", key, "edges", g.n.iloc[0], "->", list(zip(s.band, s.coef.round(4), s.p_bh.round(4))))

**Result (figs38a): not reproduced.** In every channel reading, preinfusion phase locking is higher
among high-persistence edges than among low ones in theta and alpha (BH-corrected p from below
0.001 to 0.033); with the Methods' channel set (white matter excluded) and the code's bad-channel
list also in low and high gamma (p = 0.042 and 0.012), and with white matter kept also in high
gamma. The coefficients are small (0.009 to 0.027) but the claim of no difference in any band does
not hold. The edge counts (368 and 462 with white matter excluded, 792 and 896 with it kept)
differ from the paper's 716, so the site selection differs from the paper's in a way the paper and
code do not settle (the code reads the edges from a precomputed file).

In [ ]:
print("figs38b (ketamine - preinfusion change, high vs low)")
for key, g in T38[T38.outcome.isin(["sync_change", "sync_change_z"])].groupby(["wm_excluded", "bad_list", "outcome"]):
    g = g.set_index("band")
    hold = {b: bool(g.loc[b, "p_bh"] < 0.05 and g.loc[b, "coef"] < 0) for b in ("alpha", "beta", "low gamma")}
    print(" ", key, "| alpha, beta, low gamma hold:", hold,
          "| beta coef", round(g.loc["beta", "coef"], 4), "CI", round(g.loc["beta", "ci_low"], 4), round(g.loc["beta", "ci_high"], 4),
          "in tolerance", bool(-0.057 <= g.loc["beta", "coef"] <= -0.019))

**Result (figs38b): partially reproduced.** With the change in z units (the figure's units) the
ketamine decrease is larger among high edges in beta in every channel reading (coefficients
-0.040 to -0.063, BH-corrected p from 0.001 to 0.027), within the tolerance of -0.038 ± 0.019
in 3 of the 4 readings (-0.045 with white matter excluded and the code's bad-channel list). Alpha
holds in 3 of the 4 readings (not with white matter excluded and the code's list, p = 0.057). Low
gamma, the band Table S3 calls gamma, is not significant in any reading (positive coefficients),
so the three-band claim holds only in part. The edge counts differ from the paper's 716, as for
figs38a.

## figs38h-prepuff-coupling-predicts-persistence

> Before infusion, single-trial pre-puff beta phase locking predicts post-puff persistent activity
> (and affective eye closure) across participants; during ketamine it does not.

Criterion: per trial, beta PLV averaged over all edges between kept channels in the 0.5 s before the
puff; persistent activity the single-trial projection of beta power on the persistent weights,
averaged over 0.5 to 1.5 s; affective eye closure per trial from the behaviour file; mixed-effects
model with a random intercept per participant. Reproduced when both preinfusion coefficients are
positive with p < 0.05 and both infusion coefficients have p >= 0.05. The code z-scores the
single-trial projections over all trials and does not subtract each trial's pre-puff mean (the line
is commented out, `compute_all_cd`); the Methods describe a baseline subtraction over 0.5 to 0 s
before the puff for the projection. Both are run, with and without the code's per-trial 2-s baseline.

In [ ]:
rows = []
for bad in ("code", "none"):
    for wm_plv in (False, True):
        for reading in ("code", "methods"):
            for tb in (True, False):
                for sub in (False, True):
                    df = ic.trial_table(RES, BEH, wm_plv, True, bad, reading, tb, sub)
                    for c in ("preinfusion", "infusion"):
                        d = df[df.cond == c]
                        for y in ("persistence", "eye"):
                            m = ic.mixed(d, f"{y} ~ plv", "plv")
                            rows.append(dict(bad_list=bad, plv_wm_kept=not wm_plv, open_c=reading, trial_baseline=tb,
                                             prepuff_sub=sub, cond=c, outcome=y, **m))
T38H = pd.DataFrame(rows)
print(T38H.round(4).to_string())
key = ["bad_list", "plv_wm_kept", "open_c", "trial_baseline", "prepuff_sub"]
print()
for k, g in T38H.groupby(key):
    g = g.set_index(["cond", "outcome"])
    ok = all(g.loc[("preinfusion", y), "coef"] > 0 and g.loc[("preinfusion", y), "p"] < 0.05 for y in ("persistence", "eye")) and \
         all(g.loc[("infusion", y), "p"] >= 0.05 for y in ("persistence", "eye"))
    print(k, "criterion met:", ok)
prim = T38H[(T38H.bad_list == "code") & T38H.plv_wm_kept & (T38H.open_c == "code") & T38H.trial_baseline & T38H.prepuff_sub]
print("\nmain reading (code bad list, PLV over all kept channels, Open C code, trial baseline, pre-puff subtraction):")
print(prim[["cond", "outcome", "coef", "ci_low", "ci_high", "p", "n"]].round(4).to_string(index=False))

**Result: reproduced, resting on the per-trial pre-puff subtraction.** In the main reading (214
trials of 7 participants) pre-puff beta phase locking predicts persistent activity before infusion
(coefficient 12.38, 95% CI 3.09 to 21.68, p = 0.009; the paper 12.12, p = 0.0006) and affective eye
closure (2.57, p = 0.003; the paper 3.35), and neither during ketamine (p = 0.35 and 0.98).
Eye closure behaves this way in every reading. For persistent activity the criterion depends on
the readings: with the code's bad-channel list it is met in all 8 readings that subtract each
trial's mean over 0.5 to 0 s before the puff and in 2 of the 8 that do not; without the code's
list, in 4 of 8 and 2 of 8. Without the subtraction (the code's line is commented out) the
preinfusion coefficient is mostly between 2.5 and 6 with p >= 0.05. The subtraction is adopted
because the Methods describe this baseline step for the projection and it gives the paper's
coefficient.

## ieeg-channel-atlas-link

Link claim (added by this attempt): the Yeo7 labels of the code record's `anat/pts_ch_labels.pkl`
match the dataset's channels by participant and contact label, and the record's region label of
each contact agrees with the dataset's electrode `location`.

In [ ]:
rows = []
for pt in ic.PARTICIPANTS:
    pr = RES[pt]["pairs"]
    mono = dict(zip(LABELS["pt_elec_names_mono"][pt], LABELS["pt_elec_anats_mono"][pt]))
    contacts = pd.concat([pr[["a", "loc_a"]].set_axis(["contact", "region"], axis=1),
                          pr[["b", "loc_b"]].set_axis(["contact", "region"], axis=1)]).drop_duplicates("contact")
    found = contacts.contact.isin(list(mono))
    agree = np.mean([str(mono[c]) == r for c, r in zip(contacts.contact[found], contacts.region[found])])
    bip = set(LABELS["pt_elec_names_bipo"][pt])
    rows.append(dict(pt=pt, contacts=len(contacts), contacts_matched=found.mean(), region_agreement=agree,
                     bipolar_pairs=len(pr), pairs_matched=pr.name.isin(bip).mean()))
LINK = pd.DataFrame(rows)
print(LINK.round(3).to_string(index=False))
print("all contacts and pairs matched:", bool((LINK.contacts_matched == 1).all() and (LINK.pairs_matched == 1).all()),
      "| minimum region agreement:", round(LINK.region_agreement.min(), 3))

**Result: the link holds.** Every contact and every within-shank bipolar pair of the 7 participants
is found in the code record's label file under the same name, and the record's region label agrees
with the dataset's `location` for the fraction of contacts printed above.

## figs35-persistent-dimension-distributed

> Participation in the human persistent dimension is spread uniformly across Yeo7 networks, whereas the fast dimension is over-represented in the somatomotor network.

Criterion: absolute weights divided by each participant's maximum; per Yeo7 network the median; the
labels shuffled across channels 1000 times, two-sided percentile p, BH across networks. Persistent:
reproduced when no network has corrected p < 0.05; fast: reproduced when somatomotor is
over-represented with corrected p < 0.05. Channels with a mixed or 'Depth' label take part in the
shuffle but belong to no network (as in the code).

In [ ]:
print("expected:", expected("figs35-persistent-dimension-distributed"))
for bad in ("code", "none"):
    for window in ("persistent", "fast"):
        df = ic.yeo_table(RES, LABELS, window, True, bad)
        out = ic.yeo_shuffle(df, seed=1)
        print(f"\n{window}, bad list {bad}: {len(df)} channels, {int(df.yeo7.isin(ic.YEO7).sum())} with a single Yeo7 label, "
              f"unmatched {int((df.yeo7 == 'unmatched').sum())}")
        print(out.round(3).to_string(index=False))

**Result: reproduced.** With the code's readings the channel counts per network are 87, 18, 23, 34,
50 and 63 (the paper: 88, 18, 23, 34, 50, 63). No network deviates in the persistent dimension
(smallest BH-corrected p 0.68 with the code's bad-channel list, 0.52 without), and in the fast
dimension the somatomotor median (0.446 with the code's list, the paper's 0.446) exceeds every
shuffle (corrected p = 0). The persistent frontoparietal median is 0.35, the paper's value (the
claims file's expected values are medians, the measure the legend names).

## figs37a-beta-most-similar-to-mouse

> Of the human frequency bands, the beta band power persistent projection is the most similar (highest correlation) to the mouse firing-rate persistent projection.

Criterion: Pearson correlation between the mean preinfusion persistent projection across
participants for each human band and the mean preinfusion mouse persistent projection across
sessions, over a shared peri-puff time axis from 0.5 s before to 1.5 s after the puff. The mouse
projection is computed with `mouse_npx.py` of this entry (Methods, Coding dimension, mouse
peri-puff; 13 sessions of DANDI:001326): weights from 0.5 to 1 s against the 1 s before each puff
over puff-modulated cells, projection of the preinfusion epoch average, z-scored over its 3-s window
with the 1 s before the puff subtracted. The mouse trace has 10-ms bins, so the human trace is
averaged into the same 10-ms bins.

In [ ]:
import mouse_npx
t = time.time()
MOUSE = []
for p in mouse_npx.npx_paths():
    R = mouse_npx.process_session(p)
    MOUSE.append({(var, first): mouse_npx.persist_projection(R, var=var, first_only=first)["pre"]
                  for var in ("M", "C") for first in (True, False)} | {"path": p, "n_sel": R["n_sel"]})
    del R
print(f"{len(MOUSE)} mouse sessions in {time.time() - t:.0f} s; selected cells per session:",
      [m["n_sel"] for m in MOUSE])
mt = np.arange(300) * 0.01 - 1.0          # mouse bin starts, s from the puff
sel = (mt >= -0.5 - 1e-9) & (mt < 1.5 - 1e-9)
rows = []
for open_c in ("code", "methods"):
    for wm in (True, False):
        human = {}
        for band in ic.BANDS:
            tr = ic.cd_summary(RES, ALT, band, "persistent", wm, "code", True, open_c, traces=True)["preinfusion"].mean(0)
            tt = ic.TT - 0.05 if open_c == "code" else ic.TT
            # average the 5-ms human samples into the mouse's 10-ms bins
            human[band] = np.array([tr[(tt >= a - 1e-9) & (tt < a + 0.01 - 1e-9)].mean() for a in mt[sel]])
        for var in ("M", "C"):
            for first in (True, False):
                mouse = np.mean([m[(var, first)] for m in MOUSE], axis=0)[sel]
                r = {band: st.pearsonr(human[band], mouse)[0] for band in ic.BANDS}
                best = max(r, key=r.get)
                rows.append(dict(open_c=open_c, wm_excluded=wm, mouse_var=var, mouse_first_puff=first,
                                 best=best, **{k: round(v, 3) for k, v in r.items()}))
T37A = pd.DataFrame(rows)
print(T37A.to_string(index=False))
print("readings where beta has the highest correlation:", int((T37A.best == "beta").sum()), "of", len(T37A))

**Result: reproduced, resting on the human readings of fig6c and the first-puff mouse projection.**
With the human readings that reproduce fig6c (Open C b, white matter excluded) and the mouse
persistent projection of the first puff of each series (the mouse code's choice), beta has the
highest correlation with the mouse trace (0.901 with per-neuron variances, 0.889 with the pooled
variance), ahead of broadband (0.894 and 0.871). The margin over broadband is small, and beta is
highest in 10 of the 16 readings: broadband is highest in 4 (all-puff mouse average, or white matter
kept with per-neuron variances) and low gamma in 2 (Methods z-scoring with pooled variance). All 13
sessions of DANDI:001326 were used; the paper does not say which sessions enter this comparison.
The correlation does not depend on how either trace is z-scored, only on its time course.

## Summary

| Claim | Outcome | Key numbers |
|---|---|---|
| fig6c-human-persistent-activity-reduced | reproduced | 1.363, 0.411, 0.724; t = 5.876 and 3.172 (code readings) |
| fig6e-human-fast-activity-unchanged | reproduced (assumed Open C b) | 4.120, 4.006, 3.866; t = 0.433, p = 0.680 |
| figs37c-persistent-reduction-across-bands | reproduced (assumed Open D a) | all filtered bands significant, broadband p = 0.078 |
| figs37b-fast-unchanged-across-bands | reproduced (assumed Open C b) | no corrected p < 0.05 under Open C b |
| fig6g-human-timescale-shorter | reproduced | 0.115 s and 0.092 s, t = 4.78, p = 0.003 |
| fig6i-human-coupling-reduced | partially reproduced | ratio 0.960 (p = 0.030), outside 0.979 ± 0.0061 in every reading |
| figs38a-baseline-coupling-not-different | not reproduced | theta and alpha differ at baseline in every reading |
| figs38b-ketamine-coupling-loss-larger-at-persistent-sites | partially reproduced | beta -0.045 (z) within tolerance; low gamma never significant |
| figs38h-prepuff-coupling-predicts-persistence | reproduced (assumed pre-puff subtraction) | 12.38 (p = 0.009) before, p = 0.35 during ketamine |
| ieeg-channel-atlas-link | holds | all contacts and pairs matched |
| figs35-persistent-dimension-distributed | reproduced | somatomotor fast median 0.446 |
| figs37a-beta-most-similar-to-mouse | reproduced (assumed readings) | beta r = 0.901, broadband 0.894; beta highest in 10 of 16 readings |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning
and the differences from the paper's method.

Differences from the paper's method: the bad-channel list is read from the code record, as the
dataset does not mark bad channels, and every claim is also shown without it; bipolar pairs are
formed within each electrode group; the 60-Hz notch uses Q = 20 (Methods) where the code uses 30;
the phase-locking windows are the 10 min before each eyepuff session (Table S3), as the 25-min
windows of the Methods are not in the dataset; the 5-s PLV windows do not overlap; periods of
epileptic or sleep activity are not excluded; the fig. S38 edge sets are formed here, while the
code reads them from a precomputed file.

About the dataset: bad channels and Yeo7 labels are not in it; the TTL channel is named 'CCEP'; the
LFP unit attribute says volts while the values are digitiser units with a scale factor in a comment;
the electrode coordinates are all NaN; SD245's coding-dimension preinfusion is its Baseline file in
the code, though a Preinfusion file exists.

In [ ]:
print(f"notebook run time {(time.time() - T0) / 60:.1f} min")